In [2]:
import pandas as pd
import cv2
import os
import ast
from tqdm import tqdm
import shutil

# CẤU HÌNH
INPUT_CSV = 'dataset_train.csv' # Chạy lần lượt cho train/val/test
OUTPUT_CSV = 'dataset_train_resized.csv'
OUTPUT_DIR = 'J://XQUANGRESIZE/train' # Folder lưu ảnh mới
IMG_SIZE = 256
ROOT_DIR = 'J://'

def process_and_save(input_csv, output_csv, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    df = pd.read_csv(input_csv)
    
    new_filepaths = []
    
    print(f"🚀 Bắt đầu resize {len(df)} studies từ {input_csv}...")
    
    # Duyệt qua từng dòng
    for idx, row in tqdm(df.iterrows(), total=len(df)):
        # Parse đường dẫn ảnh gốc
        original_paths = ast.literal_eval(row['filepath']) if isinstance(row['filepath'], str) else row['filepath']
        
        resized_paths_for_study = []
        
        for i, img_path in enumerate(original_paths):
            if os.path.exists(ROOT_DIR + img_path):
                # Tạo tên file mới: STUDYID_IMGINDEX.jpg
                study_id = str(row['ketqua_ID'])
                fname = f"{study_id}_{i}.jpg"
                save_path = os.path.join(output_dir, fname)
                
                # Nếu file chưa tồn tại thì mới resize (để lỡ đứt mạng chạy lại cho nhanh)
                if not os.path.exists(save_path):
                    try:
                        img = cv2.imread(ROOT_DIR + img_path)
                        if img is not None:
                            # Resize xuống 256x256
                            img_resized = cv2.resize(img, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
                            # Lưu ảnh nén JPG chất lượng 90 (nhẹ hều)
                            cv2.imwrite(save_path, img_resized, [int(cv2.IMWRITE_JPEG_QUALITY), 90])
                            resized_paths_for_study.append(save_path)
                        else:
                            # Ảnh lỗi -> Bỏ qua hoặc giữ nguyên logic xử lý sau
                            pass
                    except Exception as e:
                        print(f"Lỗi ảnh {img_path}: {e}")
                else:
                    resized_paths_for_study.append(save_path)
            else:
                pass # File gốc không thấy
        
        # Lưu lại list đường dẫn mới
        new_filepaths.append(resized_paths_for_study)
    
    # Cập nhật DataFrame và lưu CSV mới
    df['filepath'] = new_filepaths
    # Lọc bỏ những dòng không có ảnh nào sau khi resize
    df = df[df['filepath'].map(len) > 0]
    
    df.to_csv(output_csv, index=False)
    print(f"✅ Đã xong! File CSV mới: {output_csv}")

if __name__ == "__main__":
    # Resize tập Train
    process_and_save('dataset_train.csv', 
                     'dataset_train_resized.csv', 
                     'C://XQUANGRESIZE/train')
    
    # Resize tập Val (Làm luôn cho đồng bộ)
    process_and_save('dataset_val.csv', 
                     'dataset_val_resized.csv', 
                     'C://XQUANGRESIZE/val')
    
    # Resize tập Test
    process_and_save('dataset_test.csv', 
                     'dataset_test_resized.csv', 
                     'C://XQUANGRESIZE/test')

🚀 Bắt đầu resize 13387 studies từ dataset_train.csv...


100%|██████████| 13387/13387 [01:44<00:00, 128.07it/s]


✅ Đã xong! File CSV mới: dataset_train_resized.csv
🚀 Bắt đầu resize 1913 studies từ dataset_val.csv...


100%|██████████| 1913/1913 [00:06<00:00, 278.92it/s]


✅ Đã xong! File CSV mới: dataset_val_resized.csv
🚀 Bắt đầu resize 3825 studies từ dataset_test.csv...


100%|██████████| 3825/3825 [00:08<00:00, 466.91it/s] 

✅ Đã xong! File CSV mới: dataset_test_resized.csv


In [8]:
import pandas as pd
import ast
# Change image paths in CSV files from C://XQUANGRESIZE/{split}\\{STUDYID}_{IMGINDEX}.jpg to C:\XQUANGRESIZE/{split}/{STUDYID}_{IMGINDEX}.jpg
def fix_image_paths_in_csv(input_csv, output_csv):
    df = pd.read_csv(input_csv)
    
    fixed_filepaths = []
    
    for idx, row in df.iterrows():
        original_paths = ast.literal_eval(row['filepath']) if isinstance(row['filepath'], str) else row['filepath']
        fixed_paths_for_study = []
        
        for img_path in original_paths:
            # Sửa đường dẫn
            fixed_path = img_path.replace('\\', '/')
            fixed_path = fixed_path.replace('C://XQUANGRESIZE', "XQUANGRESIZE")
            fixed_paths_for_study.append(fixed_path)
        
        fixed_filepaths.append(fixed_paths_for_study)
    
    df['filepath'] = fixed_filepaths
    df.to_csv(output_csv, index=False)
    print(f"✅ Đã sửa xong file CSV: {output_csv}")

# Sửa đường dẫn trong các file CSV đã tạo
fix_image_paths_in_csv('dataset_train_resized.csv', 'dataset_train_resized_fixed.csv')
fix_image_paths_in_csv('dataset_val_resized.csv', 'dataset_val_resized_fixed.csv')
fix_image_paths_in_csv('dataset_test_resized.csv', 'dataset_test_resized_fixed.csv')

✅ Đã sửa xong file CSV: dataset_train_resized_fixed.csv
✅ Đã sửa xong file CSV: dataset_val_resized_fixed.csv
✅ Đã sửa xong file CSV: dataset_test_resized_fixed.csv
